# Install KaggleHub

In [ ]:
!pip install -q --upgrade kagglehub

# Imports

In [29]:
from pathlib import Path
from collections import Counter, defaultdict
import os
import json
import shutil
import random
import yaml
import math
from tqdm.auto import tqdm

print("Imports completed.")

Imports completed.


# Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


# Define PATHS

In [ ]:
DATASETS_ROOT = Path(
    "/content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets"
)

RAW_ROOT = DATASETS_ROOT / "raw"
PROCESSED_ROOT = DATASETS_ROOT / "processed"

BDD_RAW_DIR = RAW_ROOT / "BDD100K"
BDD_PROCESSED_DIR = PROCESSED_ROOT / "BDD100K"

YOLO_ROOT = BDD_PROCESSED_DIR / "yolo"
COCO_ROOT = BDD_PROCESSED_DIR / "coco"
METADATA_ROOT = BDD_PROCESSED_DIR / "metadata"

for directory in [
    RAW_ROOT,
    PROCESSED_ROOT,
    BDD_RAW_DIR,
    BDD_PROCESSED_DIR,
    YOLO_ROOT,
    COCO_ROOT,
    METADATA_ROOT,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Datasets root:", DATASETS_ROOT)
print("Raw BDD100K:", BDD_RAW_DIR)
print("Processed BDD100K:", BDD_PROCESSED_DIR)

Datasets root: /content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets
Raw BDD100K: /content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/raw/BDD100K
Processed BDD100K: /content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed/BDD100K


# Download dataset

In [ ]:
import kagglehub
from pathlib import Path

DATASET_HANDLE = "solesensei/solesensei_bdd100k"

dataset_path = kagglehub.dataset_download(DATASET_HANDLE)

DATASET_ROOT = Path(dataset_path)

print("Dataset downloaded.")
print("Dataset root:", DATASET_ROOT)

Using Colab cache for faster access to the 'solesensei_bdd100k' dataset.
Dataset downloaded.
Dataset root: /kaggle/input/solesensei_bdd100k


### Arborescence

In [ ]:
def show_tree(
    root: Path,
    max_depth: int = 4,
    max_items: int = 15
) -> None:
    root = Path(root)

    if not root.exists():
        raise FileNotFoundError(f"Folder not found: {root}")

    print(root)

    for current_root, directories, files in os.walk(root):
        current_path = Path(current_root)
        depth = len(current_path.relative_to(root).parts)

        if depth >= max_depth:
            directories[:] = []
            continue

        directories.sort()
        files.sort()

        indent = "    " * (depth + 1)

        for directory in directories[:max_items]:
            print(f"{indent}[DIR] {directory}")

        for filename in files[:max_items]:
            print(f"{indent}[FILE] {filename}")

        if len(directories) > max_items:
            print(
                f"{indent}... "
                f"{len(directories) - max_items} additional folders"
            )

        if len(files) > max_items:
            print(
                f"{indent}... "
                f"{len(files) - max_items} additional files"
            )


show_tree(DATASET_ROOT, max_depth=4)

/kaggle/input/solesensei_bdd100k
    [DIR] bdd100k
    [DIR] bdd100k_labels_release
    [DIR] bdd100k_seg
        [DIR] bdd100k
            [DIR] images
                [DIR] 100k
                [DIR] 10k
        [DIR] bdd100k
            [DIR] labels
                [FILE] bdd100k_labels_images_train.json
                [FILE] bdd100k_labels_images_val.json
        [DIR] bdd100k
            [DIR] seg
                [DIR] color_labels
                [DIR] images
                [DIR] labels


### Path of BDDK

In [10]:
IMAGE_ROOT = (
    DATASET_ROOT
    / "bdd100k"
    / "bdd100k"
    / "images"
)

LABEL_ROOT = (
    DATASET_ROOT
    / "bdd100k_labels_release"
    / "bdd100k"
    / "labels"
)

TRAIN_IMAGE_DIR = IMAGE_ROOT / "100k" / "train"
VAL_IMAGE_DIR = IMAGE_ROOT / "100k" / "val"
TEST_IMAGE_DIR = IMAGE_ROOT / "100k" / "test"

TRAIN_JSON = (
    LABEL_ROOT
    / "bdd100k_labels_images_train.json"
)

VAL_JSON = (
    LABEL_ROOT
    / "bdd100k_labels_images_val.json"
)

print("Train image directory:", TRAIN_IMAGE_DIR)
print("Validation image directory:", VAL_IMAGE_DIR)
print("Test image directory:", TEST_IMAGE_DIR)

print("\nTrain labels:", TRAIN_JSON)
print("Validation labels:", VAL_JSON)

Train image directory: /kaggle/input/solesensei_bdd100k/bdd100k/bdd100k/images/100k/train
Validation image directory: /kaggle/input/solesensei_bdd100k/bdd100k/bdd100k/images/100k/val
Test image directory: /kaggle/input/solesensei_bdd100k/bdd100k/bdd100k/images/100k/test

Train labels: /kaggle/input/solesensei_bdd100k/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_train.json
Validation labels: /kaggle/input/solesensei_bdd100k/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_val.json


### Check path

In [11]:
paths_to_check = {
    "Train images": TRAIN_IMAGE_DIR,
    "Validation images": VAL_IMAGE_DIR,
    "Test images": TEST_IMAGE_DIR,
    "Train JSON": TRAIN_JSON,
    "Validation JSON": VAL_JSON,
}

for name, path in paths_to_check.items():
    print(f"{name:25s}: {path.exists()} | {path}")

Train images             : True | /kaggle/input/solesensei_bdd100k/bdd100k/bdd100k/images/100k/train
Validation images        : True | /kaggle/input/solesensei_bdd100k/bdd100k/bdd100k/images/100k/val
Test images              : True | /kaggle/input/solesensei_bdd100k/bdd100k/bdd100k/images/100k/test
Train JSON               : True | /kaggle/input/solesensei_bdd100k/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_train.json
Validation JSON          : True | /kaggle/input/solesensei_bdd100k/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_val.json


# Count images


In [13]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

def count_images_recursive(directory: Path) -> int:
    if not directory.exists():
        return 0

    return sum(
        1
        for path in directory.rglob("*")
        if path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    )


print(
    "Train images:",
    f"{count_images_recursive(TRAIN_IMAGE_DIR):,}"
)

print(
    "Validation images:",
    f"{count_images_recursive(VAL_IMAGE_DIR):,}"
)

print(
    "Test images:",
    f"{count_images_recursive(TEST_IMAGE_DIR):,}"
)

Train images: 70,000
Validation images: 10,000
Test images: 20,000


In [14]:
for name, directory in {
    "train": TRAIN_IMAGE_DIR,
    "val": VAL_IMAGE_DIR,
    "test": TEST_IMAGE_DIR,
}.items():

    subfolders = [
        path
        for path in directory.iterdir()
        if path.is_dir()
    ]

    print(f"\n{name}:")
    print("Subfolders:", subfolders[:20])


train:
Subfolders: [PosixPath('/kaggle/input/solesensei_bdd100k/bdd100k/bdd100k/images/100k/train/testB'), PosixPath('/kaggle/input/solesensei_bdd100k/bdd100k/bdd100k/images/100k/train/testA'), PosixPath('/kaggle/input/solesensei_bdd100k/bdd100k/bdd100k/images/100k/train/trainB'), PosixPath('/kaggle/input/solesensei_bdd100k/bdd100k/bdd100k/images/100k/train/trainA')]

val:
Subfolders: []

test:
Subfolders: [PosixPath('/kaggle/input/solesensei_bdd100k/bdd100k/bdd100k/images/100k/test/testB'), PosixPath('/kaggle/input/solesensei_bdd100k/bdd100k/bdd100k/images/100k/test/testA'), PosixPath('/kaggle/input/solesensei_bdd100k/bdd100k/bdd100k/images/100k/test/trainB'), PosixPath('/kaggle/input/solesensei_bdd100k/bdd100k/bdd100k/images/100k/test/trainA')]


# Load annotations

In [15]:
def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)


train_data = load_json(TRAIN_JSON)
val_data = load_json(VAL_JSON)

print("Training annotation records:", f"{len(train_data):,}")
print("Validation annotation records:", f"{len(val_data):,}")

Training annotation records: 69,863
Validation annotation records: 10,000


# Check annotation structure

In [16]:
def validate_annotation_structure(records, split_name: str):
    if not isinstance(records, list):
        raise TypeError(
            f"{split_name} annotations must be a list."
        )

    if not records:
        raise ValueError(
            f"{split_name} annotations are empty."
        )

    required_keys = {"name", "attributes", "labels"}
    missing_keys = required_keys - set(records[0].keys())

    if missing_keys:
        raise KeyError(
            f"{split_name} annotations are missing: {missing_keys}"
        )


validate_annotation_structure(train_data, "Train")
validate_annotation_structure(val_data, "Validation")

print("Annotation structure is valid.")

print("\nExample record:")
print(json.dumps(train_data[0], indent=2)[:3000])

Annotation structure is valid.

Example record:
{
  "name": "0000f77c-6257be58.jpg",
  "attributes": {
    "weather": "clear",
    "scene": "city street",
    "timeofday": "daytime"
  },
  "timestamp": 10000,
  "labels": [
    {
      "category": "traffic light",
      "attributes": {
        "occluded": false,
        "truncated": false,
        "trafficLightColor": "green"
      },
      "manualShape": true,
      "manualAttributes": true,
      "box2d": {
        "x1": 1125.902264,
        "y1": 133.184488,
        "x2": 1156.978645,
        "y2": 210.875445
      },
      "id": 0
    },
    {
      "category": "traffic light",
      "attributes": {
        "occluded": false,
        "truncated": false,
        "trafficLightColor": "green"
      },
      "manualShape": true,
      "manualAttributes": true,
      "box2d": {
        "x1": 1156.978645,
        "y1": 136.637417,
        "x2": 1191.50796,
        "y2": 210.875443
      },
      "id": 1
    },
    {
      "category": "tra

# Examine category

In [17]:
def count_categories(records):
    counts = Counter()

    for record in records:
        for label in record.get("labels", []):
            category = label.get("category")

            if category:
                counts[str(category).lower()] += 1

    return counts


train_category_counts = count_categories(train_data)
val_category_counts = count_categories(val_data)

print("Training categories:\n")

for category, count in train_category_counts.most_common():
    print(f"{category:20s}: {count:,}")

print("\nValidation categories:\n")

for category, count in val_category_counts.most_common():
    print(f"{category:20s}: {count:,}")

Training categories:

car                 : 713,211
lane                : 528,643
traffic sign        : 239,686
traffic light       : 186,117
drivable area       : 125,723
person              : 91,349
truck               : 29,971
bus                 : 11,672
bike                : 7,210
rider               : 4,517
motor               : 3,002
train               : 136

Validation categories:

car                 : 102,506
lane                : 75,730
traffic sign        : 34,908
traffic light       : 26,885
drivable area       : 17,981
person              : 13,262
truck               : 4,245
bus                 : 1,597
bike                : 1,007
rider               : 649
motor               : 452
train               : 15


# Meteo condition

In [18]:
def count_weather(records):
    counts = Counter()

    for record in records:
        weather = str(
            record
            .get("attributes", {})
            .get("weather", "undefined")
        ).lower()

        counts[weather] += 1

    return counts


train_weather_counts = count_weather(train_data)
val_weather_counts = count_weather(val_data)

print("Training weather distribution:\n")

for weather, count in train_weather_counts.most_common():
    print(f"{weather:20s}: {count:,}")

print("\nValidation weather distribution:\n")

for weather, count in val_weather_counts.most_common():
    print(f"{weather:20s}: {count:,}")

Training weather distribution:

clear               : 37,344
overcast            : 8,770
undefined           : 8,119
snowy               : 5,549
rainy               : 5,070
partly cloudy       : 4,881
foggy               : 130

Validation weather distribution:

clear               : 5,346
overcast            : 1,239
undefined           : 1,157
snowy               : 769
rainy               : 738
partly cloudy       : 738
foggy               : 13


# Define classes

In [19]:
TARGET_CLASSES = [
    "person",
    "bicycle",
    "car",
    "motorcycle",
    "bus",
    "truck",
]

CLASS_TO_ID = {
    class_name: class_id
    for class_id, class_name in enumerate(TARGET_CLASSES)
}

BDD_CLASS_MAPPING = {
    "person": "person",
    "bike": "bicycle",
    "car": "car",
    "motor": "motorcycle",
    "bus": "bus",
    "truck": "truck",
}

print("Target classes:")
print(CLASS_TO_ID)

Target classes:
{'person': 0, 'bicycle': 1, 'car': 2, 'motorcycle': 3, 'bus': 4, 'truck': 5}


# Define meteo conditions

In [20]:
WEATHER_MAPPING = {
    "rainy": "rain",
    "snowy": "snow",
    "foggy": "fog",
}

TARGET_WEATHERS = {
    "rain",
    "snow",
    "fog",
}

# Filter adverse meteo conditions


In [21]:
def normalize_weather(record):
    raw_weather = str(
        record
        .get("attributes", {})
        .get("weather", "")
    ).lower()

    return WEATHER_MAPPING.get(raw_weather)


def filter_adverse_weather(records):
    filtered = []

    for record in records:
        normalized_weather = normalize_weather(record)

        if normalized_weather in TARGET_WEATHERS:
            copied_record = dict(record)
            copied_record["_weather"] = normalized_weather
            filtered.append(copied_record)

    return filtered


filtered_train_records = filter_adverse_weather(train_data)
filtered_val_records = filter_adverse_weather(val_data)

print(
    "Filtered training records:",
    f"{len(filtered_train_records):,}"
)

print(
    "Filtered validation records:",
    f"{len(filtered_val_records):,}"
)

print("\nTraining weather distribution:")
print(Counter(r["_weather"] for r in filtered_train_records))

print("\nValidation weather distribution:")
print(Counter(r["_weather"] for r in filtered_val_records))

Filtered training records: 10,749
Filtered validation records: 1,520

Training weather distribution:
Counter({'snow': 5549, 'rain': 5070, 'fog': 130})

Validation weather distribution:
Counter({'snow': 769, 'rain': 738, 'fog': 13})


# Create recursive index

In [22]:
def create_image_index(directory: Path):
    image_index = {}

    for path in directory.rglob("*"):
        if (
            path.is_file()
            and path.suffix.lower() in IMAGE_EXTENSIONS
        ):
            if path.name in image_index:
                raise ValueError(
                    f"Duplicate image filename detected: {path.name}"
                )

            image_index[path.name] = path

    return image_index


train_image_index = create_image_index(TRAIN_IMAGE_DIR)
val_image_index = create_image_index(VAL_IMAGE_DIR)

print("Indexed train images:", f"{len(train_image_index):,}")
print("Indexed validation images:", f"{len(val_image_index):,}")

Indexed train images: 70,000
Indexed validation images: 10,000


# Check missing images

In [23]:
def find_missing_images(records, image_index):
    return [
        record["name"]
        for record in records
        if record["name"] not in image_index
    ]


missing_train = find_missing_images(
    filtered_train_records,
    train_image_index
)

missing_val = find_missing_images(
    filtered_val_records,
    val_image_index
)

print("Missing training images:", len(missing_train))
print("Missing validation images:", len(missing_val))

if missing_train:
    print("Training examples:", missing_train[:10])

if missing_val:
    print("Validation examples:", missing_val[:10])

Missing training images: 0
Missing validation images: 0


# Use every adverses images

In [24]:
USE_SUBSET = False

selected_train_records = filtered_train_records
selected_val_records = filtered_val_records

print(
    "Selected training records:",
    f"{len(selected_train_records):,}"
)

print(
    "Selected validation records:",
    f"{len(selected_val_records):,}"
)

print("\nSelected training weather:")
print(Counter(r["_weather"] for r in selected_train_records))

print("\nSelected validation weather:")
print(Counter(r["_weather"] for r in selected_val_records))

Selected training records: 10,749
Selected validation records: 1,520

Selected training weather:
Counter({'snow': 5549, 'rain': 5070, 'fog': 130})

Selected validation weather:
Counter({'snow': 769, 'rain': 738, 'fog': 13})


# **YOLO**

### Prepare YOLO Folder

In [25]:
YOLO_TRAIN_IMAGES = YOLO_ROOT / "images" / "train"
YOLO_VAL_IMAGES = YOLO_ROOT / "images" / "val"

YOLO_TRAIN_LABELS = YOLO_ROOT / "labels" / "train"
YOLO_VAL_LABELS = YOLO_ROOT / "labels" / "val"

for directory in [
    YOLO_TRAIN_IMAGES,
    YOLO_VAL_IMAGES,
    YOLO_TRAIN_LABELS,
    YOLO_VAL_LABELS,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("YOLO directories created.")

YOLO directories created.


### Convert a box to YOLO format

In [26]:
def convert_box_to_yolo(
    x1,
    y1,
    x2,
    y2,
    image_width,
    image_height
):
    x1 = max(0.0, min(float(x1), image_width))
    y1 = max(0.0, min(float(y1), image_height))
    x2 = max(0.0, min(float(x2), image_width))
    y2 = max(0.0, min(float(y2), image_height))

    box_width = x2 - x1
    box_height = y2 - y1

    if box_width <= 0 or box_height <= 0:
        return None

    center_x = x1 + box_width / 2
    center_y = y1 + box_height / 2

    return (
        center_x / image_width,
        center_y / image_height,
        box_width / image_width,
        box_height / image_height,
    )

### YOLO export function

In [27]:
def export_yolo_split(
    records,
    image_index,
    output_image_directory,
    output_label_directory,
    split_name
):
    summary = {
        "split": split_name,
        "selected_records": len(records),
        "exported_images": 0,
        "exported_boxes": 0,
        "missing_images": 0,
        "invalid_boxes": 0,
        "images_without_target_objects": 0,
        "weather_counts": Counter(),
        "class_counts": Counter(),
    }

    for record in tqdm(
        records,
        desc=f"Exporting YOLO {split_name}"
    ):
        filename = record["name"]
        source_image_path = image_index.get(filename)

        if source_image_path is None:
            summary["missing_images"] += 1
            continue

        try:
            with Image.open(source_image_path) as image:
                image_width, image_height = image.size
        except Exception as error:
            print(f"Could not open {source_image_path}: {error}")
            summary["missing_images"] += 1
            continue

        label_lines = []

        for label in record.get("labels", []):
            raw_category = str(
                label.get("category", "")
            ).lower()

            target_category = BDD_CLASS_MAPPING.get(
                raw_category
            )

            if target_category not in CLASS_TO_ID:
                continue

            box = label.get("box2d")

            if not box:
                continue

            converted_box = convert_box_to_yolo(
                box.get("x1", 0),
                box.get("y1", 0),
                box.get("x2", 0),
                box.get("y2", 0),
                image_width,
                image_height,
            )

            if converted_box is None:
                summary["invalid_boxes"] += 1
                continue

            center_x, center_y, width, height = converted_box
            class_id = CLASS_TO_ID[target_category]

            label_lines.append(
                f"{class_id} "
                f"{center_x:.6f} "
                f"{center_y:.6f} "
                f"{width:.6f} "
                f"{height:.6f}"
            )

            summary["exported_boxes"] += 1
            summary["class_counts"][target_category] += 1

        # Exclude images that contain none of the six target classes
        if not label_lines:
            summary["images_without_target_objects"] += 1
            continue

        shutil.copy2(
            source_image_path,
            output_image_directory / filename
        )

        output_label_path = (
            output_label_directory
            / f"{Path(filename).stem}.txt"
        )

        output_label_path.write_text(
            "\n".join(label_lines),
            encoding="utf-8"
        )

        summary["exported_images"] += 1
        summary["weather_counts"][record["_weather"]] += 1

    summary["weather_counts"] = dict(
        summary["weather_counts"]
    )

    summary["class_counts"] = dict(
        summary["class_counts"]
    )

    return summary

### Export (YOLO)

In [31]:
from PIL import Image
from tqdm.auto import tqdm

print("PIL Image and tqdm imported.")

PIL Image and tqdm imported.


In [32]:
sample_path = next(iter(train_image_index.values()))

with Image.open(sample_path) as img:
    print("Image opened successfully.")
    print("Size:", img.size)

Image opened successfully.
Size: (1280, 720)


In [33]:
import shutil

for directory in [
    YOLO_TRAIN_IMAGES,
    YOLO_VAL_IMAGES,
    YOLO_TRAIN_LABELS,
    YOLO_VAL_LABELS,
]:
    if directory.exists():
        shutil.rmtree(directory)

    directory.mkdir(parents=True, exist_ok=True)

print("YOLO output folders reset.")

YOLO output folders reset.


In [34]:
train_yolo_summary = export_yolo_split(
    selected_train_records,
    train_image_index,
    YOLO_TRAIN_IMAGES,
    YOLO_TRAIN_LABELS,
    "train"
)

val_yolo_summary = export_yolo_split(
    selected_val_records,
    val_image_index,
    YOLO_VAL_IMAGES,
    YOLO_VAL_LABELS,
    "val"
)

Exporting YOLO train:   0%|          | 0/10749 [00:00<?, ?it/s]

Exporting YOLO val:   0%|          | 0/1520 [00:00<?, ?it/s]

In [35]:
print("Training summary:")
print(json.dumps(train_yolo_summary, indent=2))

print("\nValidation summary:")
print(json.dumps(val_yolo_summary, indent=2))

Training summary:
{
  "split": "train",
  "selected_records": 10749,
  "exported_images": 10693,
  "exported_boxes": 122579,
  "missing_images": 0,
  "invalid_boxes": 0,
  "images_without_target_objects": 56,
  "weather_counts": {
    "rain": 5043,
    "snow": 5520,
    "fog": 130
  },
  "class_counts": {
    "car": 100959,
    "truck": 4880,
    "person": 13715,
    "bus": 1938,
    "bicycle": 858,
    "motorcycle": 229
  }
}

Validation summary:
{
  "split": "val",
  "selected_records": 1520,
  "exported_images": 1504,
  "exported_boxes": 17733,
  "missing_images": 0,
  "invalid_boxes": 0,
  "images_without_target_objects": 16,
  "weather_counts": {
    "rain": 730,
    "snow": 761,
    "fog": 13
  },
  "class_counts": {
    "car": 14573,
    "truck": 754,
    "person": 1963,
    "bicycle": 138,
    "bus": 265,
    "motorcycle": 40
  }
}


### Create dataset.yaml

In [36]:
dataset_yaml = {
    "path": str(YOLO_ROOT),
    "train": "images/train",
    "val": "images/val",
    "names": {
        class_id: class_name
        for class_name, class_id in CLASS_TO_ID.items()
    },
}

YOLO_YAML_PATH = YOLO_ROOT / "dataset.yaml"

with open(
    YOLO_YAML_PATH,
    "w",
    encoding="utf-8"
) as file:
    yaml.safe_dump(
        dataset_yaml,
        file,
        sort_keys=False
    )

print(YOLO_YAML_PATH.read_text())

path: /content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed/BDD100K/yolo
train: images/train
val: images/val
names:
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: bus
  5: truck



# Faster R-CNN

### Prepare COCO path

In [37]:
COCO_ANNOTATIONS_DIR = COCO_ROOT / "annotations"

COCO_ANNOTATIONS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TRAIN_COCO_JSON = (
    COCO_ANNOTATIONS_DIR / "instances_train.json"
)

VAL_COCO_JSON = (
    COCO_ANNOTATIONS_DIR / "instances_val.json"
)

print("COCO annotations directory:")
print(COCO_ANNOTATIONS_DIR)

COCO annotations directory:
/content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed/BDD100K/coco/annotations


### Export function COCO

In [38]:
def export_coco_annotations(
    records,
    exported_image_directory,
    output_json_path,
    split_name
):
    coco = {
        "info": {
            "description": (
                "BDD100K adverse-weather object detection "
                f"{split_name} split"
            )
        },
        "licenses": [],
        "images": [],
        "annotations": [],
        "categories": [
            {
                "id": class_id + 1,
                "name": class_name,
                "supercategory": "object",
            }
            for class_name, class_id in CLASS_TO_ID.items()
        ],
    }

    record_index = {
        record["name"]: record
        for record in records
    }

    image_id = 1
    annotation_id = 1

    weather_counts = Counter()
    class_counts = Counter()

    exported_images = sorted(
        path
        for path in exported_image_directory.iterdir()
        if path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    )

    for image_path in tqdm(
        exported_images,
        desc=f"Creating COCO {split_name}"
    ):
        record = record_index.get(image_path.name)

        if record is None:
            continue

        try:
            with Image.open(image_path) as image:
                image_width, image_height = image.size
        except Exception:
            continue

        current_annotations = []

        for label in record.get("labels", []):
            raw_category = str(
                label.get("category", "")
            ).lower()

            target_category = BDD_CLASS_MAPPING.get(
                raw_category
            )

            if target_category not in CLASS_TO_ID:
                continue

            box = label.get("box2d")

            if not box:
                continue

            x1 = max(
                0.0,
                min(float(box.get("x1", 0)), image_width)
            )

            y1 = max(
                0.0,
                min(float(box.get("y1", 0)), image_height)
            )

            x2 = max(
                0.0,
                min(float(box.get("x2", 0)), image_width)
            )

            y2 = max(
                0.0,
                min(float(box.get("y2", 0)), image_height)
            )

            width = x2 - x1
            height = y2 - y1

            if width <= 0 or height <= 0:
                continue

            current_annotations.append(
                {
                    "id": annotation_id,
                    "image_id": image_id,
                    "category_id": (
                        CLASS_TO_ID[target_category] + 1
                    ),
                    "bbox": [
                        x1,
                        y1,
                        width,
                        height,
                    ],
                    "area": width * height,
                    "iscrowd": 0,
                }
            )

            annotation_id += 1
            class_counts[target_category] += 1

        if not current_annotations:
            continue

        coco["images"].append(
            {
                "id": image_id,
                "file_name": image_path.name,
                "width": image_width,
                "height": image_height,
                "weather": record["_weather"],
            }
        )

        coco["annotations"].extend(
            current_annotations
        )

        weather_counts[record["_weather"]] += 1
        image_id += 1

    with open(
        output_json_path,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(coco, file)

    return {
        "split": split_name,
        "images": len(coco["images"]),
        "annotations": len(coco["annotations"]),
        "weather_counts": dict(weather_counts),
        "class_counts": dict(class_counts),
        "json_path": str(output_json_path),
    }

### Export

In [39]:
train_coco_summary = export_coco_annotations(
    selected_train_records,
    YOLO_TRAIN_IMAGES,
    TRAIN_COCO_JSON,
    "train"
)

val_coco_summary = export_coco_annotations(
    selected_val_records,
    YOLO_VAL_IMAGES,
    VAL_COCO_JSON,
    "val"
)

print("Training COCO summary:")
print(json.dumps(train_coco_summary, indent=2))

print("\nValidation COCO summary:")
print(json.dumps(val_coco_summary, indent=2))

Creating COCO train:   0%|          | 0/10693 [00:00<?, ?it/s]

Creating COCO val:   0%|          | 0/1504 [00:00<?, ?it/s]

Training COCO summary:
{
  "split": "train",
  "images": 10693,
  "annotations": 122579,
  "weather_counts": {
    "rain": 5043,
    "snow": 5520,
    "fog": 130
  },
  "class_counts": {
    "car": 100959,
    "truck": 4880,
    "person": 13715,
    "bus": 1938,
    "bicycle": 858,
    "motorcycle": 229
  },
  "json_path": "/content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed/BDD100K/coco/annotations/instances_train.json"
}

Validation COCO summary:
{
  "split": "val",
  "images": 1504,
  "annotations": 17733,
  "weather_counts": {
    "rain": 730,
    "snow": 761,
    "fog": 13
  },
  "class_counts": {
    "car": 14573,
    "truck": 754,
    "person": 1963,
    "bicycle": 138,
    "bus": 265,
    "motorcycle": 40
  },
  "json_path": "/content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed/BDD100K/coco/annotations/instances_val.json"
}


In [42]:
TRAIN_COCO_JSON = (
    COCO_ANNOTATIONS_DIR / "instances_train.json"
)

VAL_COCO_JSON = (
    COCO_ANNOTATIONS_DIR / "instances_val.json"
)

# **CHECK**

### Check Yolo images and labels

In [40]:
def count_exported_images(directory: Path):
    return sum(
        1
        for path in directory.iterdir()
        if path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    )


train_exported_images = count_exported_images(
    YOLO_TRAIN_IMAGES
)

val_exported_images = count_exported_images(
    YOLO_VAL_IMAGES
)

train_exported_labels = len(
    list(YOLO_TRAIN_LABELS.glob("*.txt"))
)

val_exported_labels = len(
    list(YOLO_VAL_LABELS.glob("*.txt"))
)

print("Training images:", train_exported_images)
print("Training labels:", train_exported_labels)

print("\nValidation images:", val_exported_images)
print("Validation labels:", val_exported_labels)

assert train_exported_images == train_exported_labels
assert val_exported_images == val_exported_labels

print("\nImage-label correspondence verified.")

Training images: 10693
Training labels: 10693

Validation images: 1504
Validation labels: 1504

Image-label correspondence verified.


### Check YOLO coordinates

In [41]:
def validate_yolo_labels(label_directory: Path):
    errors = []

    for label_path in label_directory.glob("*.txt"):
        lines = label_path.read_text(
            encoding="utf-8"
        ).splitlines()

        for line_number, line in enumerate(lines, start=1):
            parts = line.split()

            if len(parts) != 5:
                errors.append(
                    (
                        label_path.name,
                        line_number,
                        "Expected five columns"
                    )
                )
                continue

            try:
                class_id = int(parts[0])
                coordinates = [
                    float(value)
                    for value in parts[1:]
                ]
            except ValueError:
                errors.append(
                    (
                        label_path.name,
                        line_number,
                        "Invalid numeric value"
                    )
                )
                continue

            if class_id not in CLASS_TO_ID.values():
                errors.append(
                    (
                        label_path.name,
                        line_number,
                        f"Invalid class ID: {class_id}"
                    )
                )

            if not all(
                0.0 <= value <= 1.0
                for value in coordinates
            ):
                errors.append(
                    (
                        label_path.name,
                        line_number,
                        f"Coordinates outside [0, 1]: {coordinates}"
                    )
                )

    return errors


train_yolo_errors = validate_yolo_labels(
    YOLO_TRAIN_LABELS
)

val_yolo_errors = validate_yolo_labels(
    YOLO_VAL_LABELS
)

print("Training YOLO errors:", len(train_yolo_errors))
print("Validation YOLO errors:", len(val_yolo_errors))

print("\nFirst errors:")
print((train_yolo_errors + val_yolo_errors)[:10])

Training YOLO errors: 0
Validation YOLO errors: 0

First errors:
[]


### Check JSON COCO

In [47]:
for coco_path in [
    TRAIN_COCO_JSON,
    VAL_COCO_JSON,
]:
    with open(
        coco_path,
        "r",
        encoding="utf-8"
    ) as file:
        coco_data = json.load(file)

    print("\nFile:", coco_path)
    print("Images:", len(coco_data["images"]))
    print("Annotations:", len(coco_data["annotations"]))

    print(
        "Categories:",
        [
            category["name"]
            for category in coco_data["categories"]
        ]
    )


File: /content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed/BDD100K/coco/annotations/instances_train.json
Images: 10693
Annotations: 122579
Categories: ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']

File: /content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed/BDD100K/coco/annotations/instances_val.json
Images: 1504
Annotations: 17733
Categories: ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']


# SAVE

### **Save selectioned images**

In [43]:
def save_selected_records(records, output_path):
    data = [
        {
            "name": record["name"],
            "weather": record["_weather"],
        }
        for record in records
    ]

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            data,
            file,
            indent=2
        )


save_selected_records(
    selected_train_records,
    METADATA_ROOT / "selected_train_images.json"
)

save_selected_records(
    selected_val_records,
    METADATA_ROOT / "selected_val_images.json"
)

print("Selected image lists saved.")

Selected image lists saved.


### **Save preparation summary**

In [45]:
SAMPLING_SEED = 42

In [46]:
preparation_summary = {
    "source": {
        "platform": "Kaggle",
        "dataset_handle": DATASET_HANDLE,
        "dataset_root": str(DATASET_ROOT),
        "train_json": str(TRAIN_JSON),
        "val_json": str(VAL_JSON),
    },
    "configuration": {
        "target_classes": TARGET_CLASSES,
        "target_weather": sorted(TARGET_WEATHERS),
        "use_subset": USE_SUBSET,
        "train_images_per_weather": (
            TRAIN_IMAGES_PER_WEATHER
            if USE_SUBSET
            else None
        ),
        "val_images_per_weather": (
            VAL_IMAGES_PER_WEATHER
            if USE_SUBSET
            else None
        ),
        "sampling_seed": SAMPLING_SEED,
    },
    "yolo": {
        "train": train_yolo_summary,
        "val": val_yolo_summary,
        "dataset_yaml": str(YOLO_YAML_PATH),
    },
    "coco": {
        "train": train_coco_summary,
        "val": val_coco_summary,
    },
}

SUMMARY_PATH = (
    METADATA_ROOT / "preparation_summary.json"
)

with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        preparation_summary,
        file,
        indent=2
    )

print(json.dumps(preparation_summary, indent=2))

{
  "source": {
    "platform": "Kaggle",
    "dataset_handle": "solesensei/solesensei_bdd100k",
    "dataset_root": "/kaggle/input/solesensei_bdd100k",
    "train_json": "/kaggle/input/solesensei_bdd100k/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_train.json",
    "val_json": "/kaggle/input/solesensei_bdd100k/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_val.json"
  },
  "configuration": {
    "target_classes": [
      "person",
      "bicycle",
      "car",
      "motorcycle",
      "bus",
      "truck"
    ],
    "target_weather": [
      "fog",
      "rain",
      "snow"
    ],
    "use_subset": false,
    "train_images_per_weather": null,
    "val_images_per_weather": null,
    "sampling_seed": 42
  },
  "yolo": {
    "train": {
      "split": "train",
      "selected_records": 10749,
      "exported_images": 10693,
      "exported_boxes": 122579,
      "missing_images": 0,
      "invalid_boxes": 0,
      "images_without_target_objects": 56,
      "we

# Final result

In [49]:
print("BDD100K PREPARATION COMPLETED")


print("\nRaw dataset:")
print(BDD_RAW_DIR)

print("\nProcessed dataset:")
print(BDD_PROCESSED_DIR)

print("\nYOLO dataset:")
print(YOLO_ROOT)

print("\nYOLO dataset.yaml:")
print(YOLO_YAML_PATH)

print("\nCOCO training annotations:")
print(TRAIN_COCO_JSON)

print("\nCOCO validation annotations:")
print(VAL_COCO_JSON)

print("\nMetadata:")
print(METADATA_ROOT)

print("\nExported training images:")
print(train_yolo_summary["exported_images"])

print("\nExported validation images:")
print(val_yolo_summary["exported_images"])

BDD100K PREPARATION COMPLETED

Raw dataset:
/content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/raw/BDD100K

Processed dataset:
/content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed/BDD100K

YOLO dataset:
/content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed/BDD100K/yolo

YOLO dataset.yaml:
/content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed/BDD100K/yolo/dataset.yaml

COCO training annotations:
/content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed/BDD100K/coco/annotations/instances_train.json

COCO validation annotations:
/content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed/BDD100K/coco/annotations/instances_val.json

Metadata:
/content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed/BDD100K/metadata

Exported training images:
10693

Exported validation images:
1504
